In [2]:
import pandas as pd
import numpy as np

FILE_PATH = r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Raw Data\cashflow.xlsx"

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Cash Flow",
    header=1
)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (1187, 7)
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']


In [3]:
print("Duplicate IDs:", df["id"].duplicated().sum())

print(
    "Duplicate company-year:",
    df.duplicated(["company_id", "year"]).sum()
)

print("\nMissing values:")
print(df.isna().sum())

Duplicate IDs: 0
Duplicate company-year: 23

Missing values:
id                    0
company_id            0
year                  0
operating_activity    2
investing_activity    2
financing_activity    2
net_cash_flow         2
dtype: int64


In [4]:
df["calculated_net_cash_flow"] = (
    df["operating_activity"]
    + df["investing_activity"]
    + df["financing_activity"]
)

df["cash_flow_difference"] = (
    df["net_cash_flow"]
    - df["calculated_net_cash_flow"]
)

print(
    "Reconciliation failures:",
    (df["cash_flow_difference"].abs() > 0.01).sum()
)

print(
    "Maximum difference:",
    df["cash_flow_difference"].abs().max()
)

Reconciliation failures: 396
Maximum difference: 660.0


In [5]:
duplicates = df[
    df.duplicated(["company_id", "year"], keep=False)
].sort_values(["company_id", "year"])

print("Duplicate rows:", len(duplicates))

duplicates[
    [
        "company_id",
        "year",
        "operating_activity",
        "investing_activity",
        "financing_activity",
        "net_cash_flow"
    ]
].head(50)

Duplicate rows: 46


,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
13,ABB,Mar 2014,155.0,-144.0,-42.0,-31.0
24,ABB,Mar 2014,0.0,0.0,0.0,0.0
14,ABB,Mar 2015,215.0,-187.0,-58.0,-30.0
25,ABB,Mar 2015,-35.0,-1864.0,1902.0,3.0
15,ABB,Mar 2016,249.0,-77.0,-80.0,91.0
26,ABB,Mar 2016,1544.0,-816.0,-722.0,6.0
16,ABB,Mar 2017,307.0,-155.0,-90.0,62.0
27,ABB,Mar 2017,2189.0,-1730.0,-455.0,4.0
17,ABB,Mar 2018,153.0,-215.0,-102.0,-165.0
28,ABB,Mar 2018,2198.0,-3192.0,1589.0,596.0


In [6]:
recon_issues = df[
    df["cash_flow_difference"].abs() > 0.01
].copy()

recon_issues[
    [
        "company_id",
        "year",
        "operating_activity",
        "investing_activity",
        "financing_activity",
        "net_cash_flow",
        "calculated_net_cash_flow",
        "cash_flow_difference"
    ]
].head(20)

,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow,calculated_net_cash_flow,cash_flow_difference
12,ABB,Dec 2012,101.0,-59.0,-42.0,1.0,0.0,1.0
15,ABB,Mar 2016,249.0,-77.0,-80.0,91.0,92.0,-1.0
17,ABB,Mar 2018,153.0,-215.0,-102.0,-165.0,-164.0,-1.0
21,ABB,Mar 2022,948.0,-396.0,-637.0,-86.0,-85.0,-1.0
22,ABB,Mar 2023,893.0,-148.0,-639.0,107.0,106.0,1.0
28,ABB,Mar 2018,2198.0,-3192.0,1589.0,596.0,595.0,1.0
30,ABB,Mar 2020,5437.0,-5643.0,1250.0,1045.0,1044.0,1.0
31,ABB,Mar 2021,3784.0,-4009.0,-745.0,-969.0,-970.0,1.0
32,ABB,Mar 2022,4097.0,-3936.0,-235.0,-75.0,-74.0,-1.0
33,ABB,Mar 2023,3777.0,-4699.0,923.0,2.0,1.0,1.0


In [7]:
print(
    df["cash_flow_difference"]
    .abs()
    .describe()
)

count    1185.000000
mean        0.890295
std        19.168874
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max       660.000000
Name: cash_flow_difference, dtype: float64


In [8]:
dup_groups = (
    df.groupby(["company_id", "year"])
      .size()
      .reset_index(name="count")
)

dup_groups = dup_groups[dup_groups["count"] > 1]

print("Duplicate company-year groups:", len(dup_groups))
print(dup_groups)

Duplicate company-year groups: 23
     company_id      year  count
1           ABB  Mar 2014      2
2           ABB  Mar 2015      2
3           ABB  Mar 2016      2
4           ABB  Mar 2017      2
5           ABB  Mar 2018      2
6           ABB  Mar 2019      2
7           ABB  Mar 2020      2
8           ABB  Mar 2021      2
9           ABB  Mar 2022      2
10          ABB  Mar 2023      2
11          ABB  Mar 2024      2
122  BAJAJ-AUTO  Mar 2013      2
123  BAJAJ-AUTO  Mar 2014      2
124  BAJAJ-AUTO  Mar 2015      2
125  BAJAJ-AUTO  Mar 2016      2
126  BAJAJ-AUTO  Mar 2017      2
127  BAJAJ-AUTO  Mar 2018      2
128  BAJAJ-AUTO  Mar 2019      2
129  BAJAJ-AUTO  Mar 2020      2
130  BAJAJ-AUTO  Mar 2021      2
131  BAJAJ-AUTO  Mar 2022      2
132  BAJAJ-AUTO  Mar 2023      2
133  BAJAJ-AUTO  Mar 2024      2


In [9]:
financial_cols = [
    "operating_activity",
    "investing_activity",
    "financing_activity",
    "net_cash_flow"
]

dup_rows = df[
    df.duplicated(["company_id", "year"], keep=False)
].copy()

dup_rows["financial_signature"] = (
    dup_rows[financial_cols]
    .astype(str)
    .agg("|".join, axis=1)
)

duplicate_analysis = (
    dup_rows
    .groupby(["company_id", "year"])["financial_signature"]
    .nunique()
    .reset_index(name="unique_financial_versions")
)

print(duplicate_analysis)

    company_id      year  unique_financial_versions
0          ABB  Mar 2014                          2
1          ABB  Mar 2015                          2
2          ABB  Mar 2016                          2
3          ABB  Mar 2017                          2
4          ABB  Mar 2018                          2
5          ABB  Mar 2019                          2
6          ABB  Mar 2020                          2
7          ABB  Mar 2021                          2
8          ABB  Mar 2022                          2
9          ABB  Mar 2023                          2
10         ABB  Mar 2024                          2
11  BAJAJ-AUTO  Mar 2013                          1
12  BAJAJ-AUTO  Mar 2014                          1
13  BAJAJ-AUTO  Mar 2015                          1
14  BAJAJ-AUTO  Mar 2016                          1
15  BAJAJ-AUTO  Mar 2017                          1
16  BAJAJ-AUTO  Mar 2018                          1
17  BAJAJ-AUTO  Mar 2019                          1
18  BAJAJ-AU

In [10]:
conflicting_groups = duplicate_analysis[
    duplicate_analysis["unique_financial_versions"] > 1
]

print(conflicting_groups)

   company_id      year  unique_financial_versions
0         ABB  Mar 2014                          2
1         ABB  Mar 2015                          2
2         ABB  Mar 2016                          2
3         ABB  Mar 2017                          2
4         ABB  Mar 2018                          2
5         ABB  Mar 2019                          2
6         ABB  Mar 2020                          2
7         ABB  Mar 2021                          2
8         ABB  Mar 2022                          2
9         ABB  Mar 2023                          2
10        ABB  Mar 2024                          2


In [11]:
conflicting_rows = dup_rows.merge(
    conflicting_groups[["company_id", "year"]],
    on=["company_id", "year"],
    how="inner"
)

print(
    conflicting_rows[
        [
            "company_id",
            "year",
            "operating_activity",
            "investing_activity",
            "financing_activity",
            "net_cash_flow"
        ]
    ]
)

   company_id      year  operating_activity  investing_activity  \
0         ABB  Mar 2014               155.0              -144.0   
1         ABB  Mar 2015               215.0              -187.0   
2         ABB  Mar 2016               249.0               -77.0   
3         ABB  Mar 2017               307.0              -155.0   
4         ABB  Mar 2018               153.0              -215.0   
5         ABB  Mar 2019               499.0              -257.0   
6         ABB  Mar 2020               626.0              -401.0   
7         ABB  Mar 2021               727.0               -72.0   
8         ABB  Mar 2022               948.0              -396.0   
9         ABB  Mar 2023               893.0              -148.0   
10        ABB  Mar 2024              1213.0              -416.0   
11        ABB  Mar 2014                 0.0                 0.0   
12        ABB  Mar 2015               -35.0             -1864.0   
13        ABB  Mar 2016              1544.0              -816.

In [12]:
financial_cols = [
    "company_id",
    "year",
    "operating_activity",
    "investing_activity",
    "financing_activity",
    "net_cash_flow"
]

before = len(df)

df_clean = df.drop_duplicates(
    subset=financial_cols,
    keep="first"
).copy()

print("Rows before:", before)
print("Rows after:", len(df_clean))
print("Rows removed:", before - len(df_clean))

Rows before: 1187
Rows after: 1175
Rows removed: 12


In [13]:
duplicate_counts = (
    df_clean.groupby(["company_id", "year"])
    .size()
    .reset_index(name="company_year_count")
)

df_clean = df_clean.merge(
    duplicate_counts,
    on=["company_id", "year"],
    how="left"
)

df_clean["company_year_conflict"] = (
    df_clean["company_year_count"] > 1
)

print(
    df_clean["company_year_conflict"].value_counts()
)

company_year_conflict
False    1153
True       22
Name: count, dtype: int64


In [14]:
missing_cf = df_clean[
    df_clean[
        [
            "operating_activity",
            "investing_activity",
            "financing_activity",
            "net_cash_flow"
        ]
    ].isna().any(axis=1)
]

print(missing_cf)

      id company_id      year  operating_activity  investing_activity  \
475  550   HDFCLIFE  Mar 2013                 NaN                 NaN   
476  551   HDFCLIFE  Mar 2014                 NaN                 NaN   

     financing_activity  net_cash_flow  calculated_net_cash_flow  \
475                 NaN            NaN                       NaN   
476                 NaN            NaN                       NaN   

     cash_flow_difference  company_year_count  company_year_conflict  
475                   NaN                   1                  False  
476                   NaN                   1                  False  


In [15]:
df_clean["calculated_net_cash_flow"] = (
    df_clean["operating_activity"]
    + df_clean["investing_activity"]
    + df_clean["financing_activity"]
)

df_clean["cash_flow_difference"] = (
    df_clean["net_cash_flow"]
    - df_clean["calculated_net_cash_flow"]
)

In [16]:
df_clean["free_cash_flow"] = (
    df_clean["operating_activity"]
    + df_clean["investing_activity"]
)

In [17]:
print(
    df_clean[
        [
            "company_id",
            "year",
            "operating_activity",
            "investing_activity",
            "free_cash_flow"
        ]
    ].head(15)
)

   company_id      year  operating_activity  investing_activity  \
0         TCS    Mar-13             11615.0             -6038.0   
1         TCS    Mar-14             14751.0             -9452.0   
2         TCS    Mar-15             19369.0             -1807.0   
3         TCS    Mar-16             19109.0             -5010.0   
4         TCS    Mar-17             25223.0            -16895.0   
5         TCS    Mar-18             25067.0              3104.0   
6         TCS    Mar-19             28593.0              1645.0   
7         TCS    Mar-20             32369.0              8968.0   
8         TCS    Mar-21             38802.0             -7956.0   
9         TCS    Mar-22             39949.0              -738.0   
10        TCS    Mar-23             41965.0               548.0   
11        TCS    Mar-24             44338.0              6091.0   
12        ABB  Dec 2012               101.0               -59.0   
13        ABB  Mar 2014               155.0              -144.

In [18]:
print(
    "Negative FCF:",
    (df_clean["free_cash_flow"] < 0).sum()
)

print(
    "Positive FCF:",
    (df_clean["free_cash_flow"] > 0).sum()
)

print(
    "Missing FCF:",
    df_clean["free_cash_flow"].isna().sum()
)

Negative FCF: 369
Positive FCF: 802
Missing FCF: 2


In [19]:
df_clean = df_clean.drop(
    columns=[
        "calculated_net_cash_flow",
        "cash_flow_difference"
    ]
)

In [20]:
from pathlib import Path

CLEANED_DATA = Path(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data"
)

CLEANED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

output_file = CLEANED_DATA / "cash_flow_clean.csv"

df_clean.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\cash_flow_clean.csv


In [21]:
cf = pd.read_csv(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\cash_flow_clean.csv"
)

print("Shape:", cf.shape)
print("Columns:", cf.columns.tolist())

print("\nMissing values:")
print(cf.isna().sum())

print("\nDuplicate company-year:",
      cf.duplicated(["company_id", "year"]).sum())

print("\nDuplicate IDs:",
      cf["id"].duplicated().sum())

print("\nUnique companies:",
      cf["company_id"].nunique())

Shape: (1175, 10)
Columns: ['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow', 'company_year_count', 'company_year_conflict', 'free_cash_flow']

Missing values:
id                       0
company_id               0
year                     0
operating_activity       2
investing_activity       2
financing_activity       2
net_cash_flow            2
company_year_count       0
company_year_conflict    0
free_cash_flow           2
dtype: int64

Duplicate company-year: 11

Duplicate IDs: 0

Unique companies: 100


In [22]:
duplicates = cf[
    cf.duplicated(["company_id", "year"], keep=False)
].sort_values(["company_id", "year"])

print(duplicates.to_string(index=False))

 id company_id     year  operating_activity  investing_activity  financing_activity  net_cash_flow  company_year_count  company_year_conflict  free_cash_flow
 62        ABB Mar 2014               155.0              -144.0               -42.0          -31.0                   2                   True            11.0
 73        ABB Mar 2014                 0.0                 0.0                 0.0            0.0                   2                   True             0.0
 63        ABB Mar 2015               215.0              -187.0               -58.0          -30.0                   2                   True            28.0
 74        ABB Mar 2015               -35.0             -1864.0              1902.0            3.0                   2                   True         -1899.0
 64        ABB Mar 2016               249.0               -77.0               -80.0           91.0                   2                   True           172.0
 75        ABB Mar 2016              1544.0         

In [23]:
missing = cf[
    cf[[
        "operating_activity",
        "investing_activity",
        "financing_activity",
        "net_cash_flow",
        "free_cash_flow"
    ]].isna().any(axis=1)
]

print(missing.to_string(index=False))

 id company_id     year  operating_activity  investing_activity  financing_activity  net_cash_flow  company_year_count  company_year_conflict  free_cash_flow
550   HDFCLIFE Mar 2013                 NaN                 NaN                 NaN            NaN                   1                  False             NaN
551   HDFCLIFE Mar 2014                 NaN                 NaN                 NaN            NaN                   1                  False             NaN


In [24]:
raw_cf = pd.read_excel(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Raw Data\cashflow.xlsx",
    header=1
)

abb_raw = raw_cf[
    raw_cf["company_id"].astype(str).str.strip() == "ABB"
]

print(abb_raw.to_string(index=False))

 id company_id     year  operating_activity  investing_activity  financing_activity  net_cash_flow
 61        ABB Dec 2012               101.0               -59.0               -42.0            1.0
 62        ABB Mar 2014               155.0              -144.0               -42.0          -31.0
 63        ABB Mar 2015               215.0              -187.0               -58.0          -30.0
 64        ABB Mar 2016               249.0               -77.0               -80.0           91.0
 65        ABB Mar 2017               307.0              -155.0               -90.0           62.0
 66        ABB Mar 2018               153.0              -215.0              -102.0         -165.0
 67        ABB Mar 2019               499.0              -257.0              -143.0           99.0
 68        ABB Mar 2020               626.0              -401.0              -217.0            8.0
 69        ABB Mar 2021               727.0               -72.0              -582.0           73.0
 70       

In [25]:
print("Raw rows:", len(raw_cf))
print("Clean rows:", len(cf))
print("Rows removed:", len(raw_cf) - len(cf))

Raw rows: 1187
Clean rows: 1175
Rows removed: 12


In [26]:
print("\nCleaned ABB rows:", 
      len(cf[cf["company_id"] == "ABB"]))

print("\nCleaned AGTL rows:", 
      len(cf[cf["company_id"] == "AGTL"]))

print("\nCleaned VBL rows:", 
      len(cf[cf["company_id"] == "VBL"]))


Cleaned ABB rows: 23

Cleaned AGTL rows: 7

Cleaned VBL rows: 12


In [28]:
import pandas as pd

path = r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\cash_flow_mysql.csv"

cf = pd.read_csv(path)

# Recalculate free cash flow only when both source values exist
cf["free_cash_flow"] = (
    cf["operating_activity"] + cf["investing_activity"]
)

# If either source value is missing, keep FCF as missing
cf.loc[
    cf["operating_activity"].isna() |
    cf["investing_activity"].isna(),
    "free_cash_flow"
] = pd.NA

# Save with blank fields for SQL NULL conversion
cf.to_csv(path, index=False)

print("Saved:", path)
print("\nHDFCLIFE:")
print(
    cf[cf["company_id"] == "HDFCLIFE"].to_string(index=False)
)

Saved: C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\cash_flow_mysql.csv

HDFCLIFE:
 id company_id     year  operating_activity  investing_activity  financing_activity  net_cash_flow  company_year_count  company_year_conflict  free_cash_flow
550   HDFCLIFE Mar 2013                 NaN                 NaN                 NaN            NaN                   1                      0             NaN
551   HDFCLIFE Mar 2014                 NaN                 NaN                 NaN            NaN                   1                      0             NaN
552   HDFCLIFE Mar 2015              4459.0             -3514.0              -168.0          777.0                   1                      0           945.0
553   HDFCLIFE Mar 2016              5687.0             -3962.0              -212.0         1513.0                   1                      0          1725.0
554   HDFCLIFE Mar 2017              6230.0             -5177.0              -236.0          817.0  

In [30]:
# Recalculate FCF only where it is currently missing
mask = (
    cf["free_cash_flow"].isna()
    & cf["operating_activity"].notna()
    & cf["investing_activity"].notna()
)

cf.loc[mask, "free_cash_flow"] = (
    cf.loc[mask, "operating_activity"]
    + cf.loc[mask, "investing_activity"]
)

print("FCF values recalculated:", mask.sum())

print(
    cf[cf["company_id"] == "HDFCLIFE"].to_string(index=False)
)

FCF values recalculated: 0
 id company_id     year  operating_activity  investing_activity  financing_activity  net_cash_flow  company_year_count  company_year_conflict  free_cash_flow
550   HDFCLIFE Mar 2013                 NaN                 NaN                 NaN            NaN                   1                      0             NaN
551   HDFCLIFE Mar 2014                 NaN                 NaN                 NaN            NaN                   1                      0             NaN
552   HDFCLIFE Mar 2015              4459.0             -3514.0              -168.0          777.0                   1                      0           945.0
553   HDFCLIFE Mar 2016              5687.0             -3962.0              -212.0         1513.0                   1                      0          1725.0
554   HDFCLIFE Mar 2017              6230.0             -5177.0              -236.0          817.0                   1                      0          1053.0
555   HDFCLIFE Mar 2018  

In [31]:
output_path = r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\cash_flow_mysql.csv"

cf.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\cash_flow_mysql.csv
